# 04 — Instance error analysis

**Purpose:** audit v2 filament matches, misses, false detections, and overlap failures  
**Inputs:** v2 best checkpoint and deterministic validation split  
**Outputs:** diagnostic plots and counts; no files written  
**Dependencies:** project `.conda`, MAGFiLO data, trained v2 checkpoint, `src/filament_seg`.  
**Execution:** run top to bottom from the root or `notebooks/`; no training occurs.  
**Runtime:** two 2048² images take several seconds and need GPU memory  
**Related:** `02_training.ipynb` produces weights, `03_submission.ipynb` consumes selected settings.

PQ here is diagnostic and not verified against the organizer's exact matcher.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
cwd = Path.cwd().resolve()
project_root = cwd if (cwd/'src/filament_seg/dataset.py').is_file() else cwd.parent
assert (project_root/'src/filament_seg/dataset.py').is_file()
assert Path(sys.executable).resolve() == (project_root/'.conda/python.exe').resolve(), sys.executable
if str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.model import build_model
from filament_seg.train import load_checkpoint
from filament_seg.inference import predict_tiled, iter_probability_instances
from filament_seg.evaluate import diagnostic_panoptic_quality
from filament_seg.utils import get_device
device = get_device()
base = project_root/'data/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train'
dataset = SolarFilamentDataset(base/'train_images',base/'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
val_indices = np.random.default_rng(42).permutation(len(dataset))[565:].tolist()
checkpoint_path = project_root/'checkpoints/unet_resnet34_v2/best_model.pth'
if not checkpoint_path.is_file(): raise FileNotFoundError(checkpoint_path)
model = build_model(pretrained=False).to(device)
checkpoint = load_checkpoint(checkpoint_path,model,map_location=device)
assert checkpoint.get('epoch',0)>0 and not any(r.get('smoke_test') for r in checkpoint.get('history',[]))
model.eval()
print('Checkpoint epoch:',checkpoint.get('epoch'),'device:',device,'validation images:',len(val_indices))

## Native-resolution instance audit

Use one COCO annotation set per physical filename. Predictions are connected components of a semantic probability map, so touching filaments can merge. The historical 695 TP / 742 FP / 291 FN was reported at threshold 0.75 and area 300; it is a reference, not a result of the cells below.

In [ ]:
THRESHOLD, MIN_AREA = 0.75, 300
SELECTED_INDICES = val_indices[:2]  # bounded, reproducible demonstration
audit = []
for idx in SELECTED_INDICES:
    sample = dataset[idx]
    probability = predict_tiled(model, sample['image'], tile_size=512, overlap=128, device=device)
    gt = [item['mask'] for item in dataset.iter_instances(idx,0)]
    pred = list(iter_probability_instances(probability,threshold=THRESHOLD,min_area=MIN_AREA))
    metric = diagnostic_panoptic_quality(gt,pred,return_details=True)
    audit.append((sample,probability,gt,pred,metric))
    print(sample['filename'],{key:metric[key] for key in ['pq','sq','rq','tp','fp','fn']})
    print('Matched IoUs:',[round(item['iou'],3) for item in metric['matches']])
    print('Missed GT areas:',[int(gt[k].sum()) for k in metric['unmatched_gt']])
    print('False prediction areas:',[int(pred[k].sum()) for k in metric['unmatched_pred']])

## Qualitative errors

Green is matched instance foreground, red is unmatched predicted foreground, and blue is missed ground-truth foreground. This is **instance-level** matching; pixel overlap alone may disagree. Review dense touching structures, faint filaments, limb artifacts, and annotation alternatives before interpreting the counts.

In [ ]:
for sample,prob,gt,pred,metric in audit:
    image = sample['image'].squeeze().numpy()
    gt_union = np.logical_or.reduce(gt) if gt else np.zeros_like(image,bool)
    pred_union = np.logical_or.reduce(pred) if pred else np.zeros_like(image,bool)
    matched_gt = {m['gt_index'] for m in metric['matches']}
    matched_pred = {m['pred_index'] for m in metric['matches']}
    overlay = np.repeat((image*0.5)[...,None],3,axis=2)
    for k in matched_gt: overlay[gt[k].astype(bool)] = (0,1,0)
    for k in metric['unmatched_gt']: overlay[gt[k].astype(bool)] = (0,0,1)
    for k in metric['unmatched_pred']: overlay[pred[k].astype(bool)] = (1,0,0)
    fig,axes=plt.subplots(1,5,figsize=(21,4))
    for ax,arr,title in zip(axes,[image,gt_union,prob,pred_union,overlay],['Image','GT instances union','Probability','Predicted instances union','Instance TP/FP/FN']):
        ax.imshow(arr,cmap=None if arr.ndim==3 else 'gray');ax.set_title(title);ax.axis('off')
    fig.suptitle(sample['filename']+f" | PQ={metric['pq']:.3f}")
    plt.tight_layout();plt.show()

## Interpretation and next steps

The two-image sample is diagnostic only. It cannot validate or reproduce full-validation historical counts. Categorize each unmatched instance by size, limb location, touching neighbors, and annotator disagreement before attributing a dominant failure mode. Notebook 05 provides a separated tuning/holdout comparison.